In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

In [38]:
loader = PyPDFLoader("../data/ncpg.pdf")

documents = loader.load()

print("Pages:", len(documents))

Pages: 15


In [39]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 62


In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
vector = embedding_model.embed_query(
    chunks[0].page_content
)

print("Vector length:", len(vector))
print(vector[:10])

In [ ]:
from langchain_chroma import Chroma

In [ ]:
vector_store = Chroma(
    collection_name="ncpg_langchain",
    embedding_function=embedding_model,
    persist_directory="../chroma_db"
)

# which means 
# Which collection?       → ncpg_langchain
# How to create vectors?  → embedding_model
# Where to save database? → ../chroma_db

In [ ]:
vector_store.add_documents(chunks)

#under the hood
# chunks
#   ↓
# extract page_content
#   ↓
# embedding_model
#   ↓
# vectors
#   ↓
# store text
#   ↓
# store metadata
#   ↓
# store vectors in Chroma

In [ ]:
print(
    "Stored chunks:",
    vector_store._collection.count()
)

In [ ]:
query = "What does the document say about responsible gaming?"

results = vector_store.similarity_search(
    query,
    k=3
)

# it is essentially Doing the following

# query text
#    ↓
# embed query
#    ↓
# search Chroma vectors
#    ↓
# top 3
#    ↓
# return Document objects

In [ ]:
for i, doc in enumerate(results):
    print(f"\nRESULT {i + 1}")
    print(doc.page_content[:500])
    print("METADATA:", doc.metadata)
    print("-" * 60)

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

# vector_store
#     ↓
# as_retriever()
#     ↓
# Retriever object
#     ↓
# always fetch top 3 relevant chunks

In [ ]:
query = "What does the document say about responsible gaming?"

retrieved_docs = retriever.invoke(query)

In [ ]:
for i, doc in enumerate(retrieved_docs):
    print(f"\nRESULT {i + 1}")
    print(doc.page_content[:500])
    print("METADATA:", doc.metadata)
    print("-" * 60)

Before                                                           

query
  ↓
vector_store.similarity_search(...)

Now 

query
  ↓
retriever.invoke(...)

Why bother adding another object?

Because later LangChain chains are designed to work naturally with a: Retriever

So instead of your chain needing to know:
Chroma
embedding model
similarity search
k value

it can simply say: Give this question to the retriever.

Now the pipeline becomes

PDF
 ↓
PyPDFLoader
 ↓
RecursiveCharacterTextSplitter
 ↓
HuggingFaceEmbeddings
 ↓
Chroma
 ↓
Retriever   ← you are here
 ↓
Prompt
 ↓
LLM
 ↓
Answer



In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
#Creating llm

llm = ChatOpenAI(
    model="gpt-5-mini"
)

In [ ]:
#Creating Prompt

prompt = ChatPromptTemplate.from_template("""
Answer the user's question using only the context below.

If the answer cannot be found in the context, say:
"The information was not found in the provided documents."

Context:
{context}

Question:
{question}
""")

In [ ]:
question = "What does the document say about responsible gaming?"

retrieved_docs = retriever.invoke(question)

In [ ]:
context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

In [ ]:
formatted_prompt = prompt.invoke({
    "context": context,
    "question": question
})

# Prompt template
#      +
# context
#      +
# question
#      ↓
# completed prompt

In [ ]:
response = llm.invoke(formatted_prompt)
print(response.content)

# question
#    ↓
# retriever.invoke()
#    ↓
# top 3 Document objects
#    ↓
# join page_content
#    ↓
# context
#    ↓
# ChatPromptTemplate
#    ↓
# ChatOpenAI
#    ↓
# answer

In [ ]:
# importing chain imports

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [27]:
from langchain_core.runnables import RunnablePassthrough

rag_chain_with_sources = RunnablePassthrough.assign(
    context=lambda x: retriever.invoke(x["question"])
).assign(
    answer=lambda x: (
        prompt
        | llm
        | StrOutputParser()
    ).invoke({
        "context": format_docs(x["context"]),
        "question": x["question"]
    })
)

User question
      ↓
┌──────────────────────────────┐
│ context                      │
│ question → retriever         │
│          → format_docs       │
│                              │
│ question                     │
│ question → pass through      │
└──────────────────────────────┘
      ↓
prompt
      ↓
llm
      ↓
StrOutputParser
      ↓
plain string answer


In [ ]:
answer = rag_chain.invoke(
    "What does the document say about responsible gaming?"
)

print(answer)

In [31]:
result = rag_chain_with_sources.invoke({
    "question": "What does the document say about responsible gaming?"
})

In [32]:
print(result["answer"])

The document (IRGS) says that operators must have formal responsible-gambling measures, including:

- A Corporate Responsible Gambling Policy that states the company’s commitment to player protection and harm mitigation, outlines leadership expectations, designates an executive/senior staff member responsible for implementing policies (with that contact listed publicly), and is reviewed annually.
- A Responsible Gambling Strategy with defined goals, a clear action plan (timelines, risk assessments), measurable expected outcomes and metrics of success; the strategy is evaluated annually, reported publicly, and incorporates all IRGS recommendations.
- Staff training (including additional annual training for customer‑facing staff, hosts and VIP/account managers) covering topics such as myths and facts about gambling, the company’s policies and strategy, staff roles in player protection, the employee gambling policy, the regulatory problem‑gambling helpline, available responsible‑gambling 

In [33]:
for i, doc in enumerate(result["context"]):
    print(f"\nSOURCE {i + 1}")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:500])
    print("-" * 60)


SOURCE 1
Metadata: {'grammarlydocumentid': 'c464feebad53297778565d304e63b0b08789ece97dfae5aacf44e5ec98171d52', 'keywords': '', 'contenttypeid': '0x01010009622177F12D21469D2327DAA6BB999C', 'producer': 'Adobe PDF Library 26.1.183', 'creator': 'Acrobat PDFMaker 26 for Word', 'subject': '', 'comments': '', 'title': 'Internet Responsible Gambling Standards 2026', 'moddate': '2026-04-29T13:22:45-07:00', 'source': '../data/ncpg.pdf', 'page': 2, 'mediaserviceimagetags': '', 'author': 'Danielle Nekimken', 'company': '', 'sourcemodified': '', 'creationdate': '2026-04-29T13:22:43-07:00', 'page_label': '3', 'total_pages': 15}
Internet Responsible Gambling Standards 2026  Page | 3 
The IRGS presents ten categories of recommended guidelines, each with its own section in the 
document. Each category is broken down with subheadings that provide more concrete 
information on what the IRGS recommend for successful implementation. 
 
 
Governance 
 
Corporate Responsible Gambling Policy   
A Corporate R

In [34]:
sources = []

for doc in result["context"]:
    source = doc.metadata.get("source")
    page = doc.metadata.get("page")

    sources.append({
        "source": source,
        "page": page
    })

sources

[{'source': '../data/ncpg.pdf', 'page': 2},
 {'source': '../data/ncpg.pdf', 'page': 3},
 {'source': '../data/ncpg.pdf', 'page': 2}]

In [35]:
print("Page:", doc.metadata.get("page", 0) + 1)

Page: 3


Question
   ↓
Retriever
   ↓
Top-K Documents ───────────────┐
   ↓                           │
format_docs()                  │
   ↓                           │
Prompt                         │
   ↓                           │
LLM                            │
   ↓                           │
Answer                         │
                               ↓
                         Source metadata

In [40]:
print(chunks[0].metadata)

{'producer': 'Adobe PDF Library 26.1.183', 'creator': 'Acrobat PDFMaker 26 for Word', 'creationdate': '2026-04-29T13:22:43-07:00', 'author': 'Danielle Nekimken', 'comments': '', 'company': '', 'contenttypeid': '0x01010009622177F12D21469D2327DAA6BB999C', 'grammarlydocumentid': 'c464feebad53297778565d304e63b0b08789ece97dfae5aacf44e5ec98171d52', 'keywords': '', 'mediaserviceimagetags': '', 'moddate': '2026-04-29T13:22:45-07:00', 'sourcemodified': '', 'subject': '', 'title': 'Internet Responsible Gambling Standards 2026', 'source': '../data/ncpg.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


In [41]:
filtered_retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 3,
        "filter": {
            "source": "../data/ncpg.pdf"
        }
    }
)

#conceptually

Question
   ↓
Filter eligible chunks first
   ↓
only chunks where source = ncpg.pdf
   ↓
semantic similarity search
   ↓
top 3

In [42]:
question = "What does the document say about responsible gaming?"

filtered_docs = filtered_retriever.invoke(question)

for i, doc in enumerate(filtered_docs):
    print(f"\nRESULT {i + 1}")
    print("SOURCE:", doc.metadata.get("source"))
    print("PAGE:", doc.metadata.get("page"))
    print(doc.page_content[:400])
    print("-" * 60)


RESULT 1
SOURCE: ../data/ncpg.pdf
PAGE: 2
Internet Responsible Gambling Standards 2026  Page | 3 
The IRGS presents ten categories of recommended guidelines, each with its own section in the 
document. Each category is broken down with subheadings that provide more concrete 
information on what the IRGS recommend for successful implementation. 
 
 
Governance 
 
Corporate Responsible Gambling Policy   
A Corporate Responsible Gambling Pol
------------------------------------------------------------

RESULT 2
SOURCE: ../data/ncpg.pdf
PAGE: 3
Internet Responsible Gambling Standards 2026  Page | 4 
• Myths and facts about gambling 
• The company’s responsible gambling policies and strategy 
• The role each staff member/department plays in contributing to player protection and 
harm reduction 
• The employee gambling policy 
• The ‘problem gambling’ helpline specified/approved in regulations.  
• Available responsible gambling tools (inc
---------------------------------------------------

In [43]:
filtered_rag_chain = RunnablePassthrough.assign(
    context=lambda x: filtered_retriever.invoke(x["question"])
).assign(
    answer=lambda x: (
        prompt
        | llm
        | StrOutputParser()
    ).invoke({
        "context": format_docs(x["context"]),
        "question": x["question"]
    })
)

In [44]:
result = filtered_rag_chain.invoke({
    "question": "What does the document say about responsible gaming?"
})

print(result["answer"])

The document treats responsible gambling as a central governance requirement and gives detailed guidance, including:

- A Corporate Responsible Gambling Policy that:
  - Declares the company’s commitment to responsible gambling, player protection and harm mitigation across strategic directions.
  - Outlines leadership expectations and requires designation of an executive/senior staff member responsible for implementing responsible gambling policies, procedures and ongoing initiatives (their contact is published).
  - Is reviewed annually.

- A Responsible Gambling Strategy that:
  - Specifies defined goals, a clear plan of action with timelines, risk assessments, measurable expected outcomes and metrics of success.
  - Is evaluated annually, with the evaluation report made publicly available.
  - Incorporates the IRGS recommendations.

- Requirements for staff and customer information:
  - Customer-facing staff (including hosts and VIP/account managers) receive additional annual traini

In [45]:
for doc in result["context"]:
    print(
        "Source:",
        doc.metadata.get("source"),
        "| Page:",
        doc.metadata.get("page")
    )

Source: ../data/ncpg.pdf | Page: 2
Source: ../data/ncpg.pdf | Page: 3
Source: ../data/ncpg.pdf | Page: 2


### Manual RAG                       LangChain

- PdfReader                 -         PyPDFLoader
- chunk_text()              -      RecursiveCharacterTextSplitter
- SentenceTransformer       -      HuggingFaceEmbeddings
- collection.add()          -      Chroma.add_documents()
- collection.query()        -      retriever.invoke()
- where={...}               -      search_kwargs={"filter": ...}
- f-string prompt           -      ChatPromptTemplate
- OpenAI client             -      ChatOpenAI
- manual pipeline           -      LCEL / Runnable chain

In [46]:
print("Total records:", vector_store._collection.count())

Total records: 310


In [47]:
print("Original chunks:", len(chunks))

Original chunks: 62


In [48]:
data = vector_store._collection.get(
    include=["documents", "metadatas"]
)

print("Stored IDs:", len(data["ids"]))
print("Stored documents:", len(data["documents"]))

Stored IDs: 310
Stored documents: 310


In [49]:
for i in range(min(5, len(data["ids"]))):
    print("ID:", data["ids"][i])
    print("Metadata:", data["metadatas"][i])
    print(data["documents"][i][:200])
    print("-" * 60)

ID: 5b723e09-39f7-4db5-bf80-b34ff46c5b15
Metadata: {'creationdate': '2026-04-29T13:22:43-07:00', 'mediaserviceimagetags': '', 'comments': '', 'creator': 'Acrobat PDFMaker 26 for Word', 'page': 0, 'moddate': '2026-04-29T13:22:45-07:00', 'subject': '', 'sourcemodified': '', 'keywords': '', 'grammarlydocumentid': 'c464feebad53297778565d304e63b0b08789ece97dfae5aacf44e5ec98171d52', 'contenttypeid': '0x01010009622177F12D21469D2327DAA6BB999C', 'page_label': '1', 'producer': 'Adobe PDF Library 26.1.183', 'title': 'Internet Responsible Gambling Standards 2026', 'author': 'Danielle Nekimken', 'total_pages': 15, 'company': '', 'source': '../data/ncpg.pdf'}
Internet Responsible Gambling Standards 
  
Revised April 2026 
 
Introduction 
 
All forms of gambling have the potential to bring both positive and negative consequences to 
the customer. Internet g
------------------------------------------------------------
ID: 0113d2de-b5fd-4085-a19f-934074336d83
Metadata: {'mediaserviceimagetags': '', 'pa

In [50]:
from collections import Counter

document_counts = Counter(data["documents"])

duplicates = {
    doc: count
    for doc, count in document_counts.items()
    if count > 1
}

print("Unique duplicated chunks:", len(duplicates))
print("Extra duplicate records:", sum(count - 1 for count in duplicates.values()))

Unique duplicated chunks: 62
Extra duplicate records: 248


In [51]:
vector_store._client.delete_collection(
    name="ncpg_project_test"
)

print("Collection deleted.")

Collection deleted.


In [52]:
vector_store = Chroma(
    collection_name="ncpg_project_test",
    embedding_function=embedding_model,
    persist_directory="../chroma_db"
)

print("Count:", vector_store._collection.count())

Count: 0


In [53]:
vector_store.add_documents(chunks)

print("Stored:", vector_store._collection.count())
print("Expected:", len(chunks))

Stored: 62
Expected: 62


In [ ]:
# running the following line multiple times is dangerous, because it adds records again and again.
# vector_store.add_documents(chunks)

In [54]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

from rag_app.config import CHROMA_DIR


embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma(
    collection_name="ncpg_project_test",
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR)
)

vector_store._client.delete_collection(
    name="ncpg_project_test"
)

print("Deleted ncpg_project_test")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3625.49it/s]


Deleted ncpg_project_test
